# <center> 705. Design HashSet </center>


## Problem Description
[Click here](https://leetcode.com/problems/design-hashset/description/)


## Intuition
<!-- Describe your first thoughts on how to solve this problem. -->
- **Approach 1: Boolean array**
    - use an array where the index represents the key
    - there is no hashing or collision because each key has its own index
    - O(1) time for all operations, but we need to pre-allocate the size
    - for example, if the max possible value is 50, we have to create an array of size 50+1 even if the input contains a single value
- **Approach 2: Linked List**
    - use an array of buckets, with a linked list in each bucket
    - different keys can have the same hash value, so we store them in a linked list to handle collisions
    - resize the bucket array when it becomes too full to keep the linked lists short and maintain O(1) average time


## Approach
<!-- Describe your approach to solving the problem. -->
### Approach 1: Boolean Array

**init()**
- create a boolean array of size 1000001 to represent the set
    - *1000001 because the max possible value is $10^6$*
    - *$10^6$ = 1000000, the extra 1 is to keep the index in bounds*

**add()**
- set the value at the corresponding index to true

**remove()**
- set the value at the corresponding index to false

**contains()**
- check the value at the respective index and return the result
    - *if the value is true, the key is present in the set and vice versa*

### Approach 2: Linked List
*create a separate class for the linked list nodes*

#### List Node Class

**init()**
- set the node key and next pointer

#### Hashset Class

**init()**
- set hashset capacity = 1000
    - *1000 is the initial number of buckets we choose for the hashset*
    - *we can use a smaller capacity, but it will result in more collisions and longer linked lists*
- set size = 0 to track the number of keys in the hashset
- initialize an array of size 1000 with dummy linked list nodes to represent the buckets of the hashset
    - *dummy node because it makes linked list operations easier by avoiding a special case for the first node*

**hash()**
- take the mod of key to get the index and return the result

**resize()**
- save the current buckets
- double the capacity
- create a new larger bucket array using the new capacity
- for each bucket in the old buckets
    - set cur to the first actual node, skipping the dummy node
    - for each node in the current bucket
        - save the next node before moving cur
        - calculate the new bucket index for this key
            - *we need to rehash the keys because the bucket index depends on the capacity*
        - *move the current node to its new bucket*
            - connect the current node to the new bucket's first node
            - put the current node at the beginning of the new bucket
        - move to the next old node

**add()**
- find the key index and set current node pointer cur = the dummy node at the index
- loop through the linked list to check if the key already exists
    - return if the key is found
- else create a new node for the key
- increase size by 1
- if the load factor (number of keys / number of buckets) becomes too high
    - resize the hashset

**remove()**
- find index and set cur = the dummy node at the index
- loop until you find the node before the key node
- if you find the previous node
    - remove the key node by updating the next pointer of the previous node
    - decrease size by 1

**contains()**
- find index and set cur = the dummy node at the index
- loop until you find the key node
    - if the key is found
        - return true
    - move cur to the next node
- else return false


## Complexity
<!-- Add your time complexity here, e.g. $$O(n)$$ -->
- Time complexity: 
    - Boolean Array Approach:
        - init(): O(array) → O(max possible key) → O(m)
            - *m = max possible key + 1 = $10^6 + 1$ here*
        - add(): O(1)
        - remove(): O(1)
        - contains(): O(1)
        - overall time: O(m)
    - Linked List Approach:
        - init(): O(bucket array) → O(1000) → O(1) 
        - hash(): O(1)
        - resize(): O(new bucket array + moving nodes) → O(capacity × 2 + n) → O(n)
        - add(), remove() and contains(): O(1) on average and O(n) in the worst case
            - *resizing takes O(n), but it happens only occasionally, so add() is O(1) amortized*
            - *in the worst case, all keys can have the same hash value, resulting in a linked list of size n*
        - overall time: O(1) amortized

<!-- Add your space complexity here, e.g. $$O(n)$$ -->
- Space complexity: 
    - Boolean Array Approach:
        - init(): O(array) → O(max possible key) → O(m)
        - add(): O(1)
        - remove(): O(1)
        - contains(): O(1)
        - overall space: O(m)
    - Linked List Approach
        - init(): O(bucket array) → O(1000) → O(1)
        - hash(): O(1)
        - resize(): O(new bucket array) → O(B) → O(n) because B grows with n
        - add(): O(1)
        - remove(): O(1)
        - contains(): O(1)
        - overall space: O(buckets array + total nodes) → O(B + n) → O(n)
            - *after n calls to add(), the linked list can contain up to n nodes*


## Code

In [ ]:
# Approach 1: Boolean Array

class MyHashSet:

    def __init__(self):
        self.present = [False] * 1000001

    def add(self, key: int) -> None:
        self.present[key] = True

    def remove(self, key: int) -> None:
        self.present[key] = False

    def contains(self, key: int) -> bool:
        return self.present[key]



# Approach 2: Linked List

class ListNode:

    def __init__(self, key= -1, next= None):
        self.key = key
        self.next = next
        

class MyHashSet:

    def __init__(self):
        self.capacity = 1000
        self.size = 0
        self.buckets = [ListNode() for _ in range(self.capacity)]
    
    def hash(self, key: int) -> int:
        return key % self.capacity

    def resize(self):
        old_buckets = self.buckets
        self.capacity *= 2
        self.buckets = [ListNode() for _ in range(self.capacity)]
        for head in old_buckets:
            cur = head.next
            while cur:
                next_node = cur.next
                index = self.hash(cur.key)
                cur.next = self.buckets[index].next
                self.buckets[index].next = cur
                cur = next_node
        
    def add(self, key: int) -> None:
        cur = self.buckets[self.hash(key)]
        while cur.next:
            if cur.next.key == key:
                return
            cur = cur.next
        cur.next = ListNode(key)
        self.size += 1
        if self.size / self.capacity > 0.75:
            self.resize()

    def remove(self, key: int) -> None:
        cur = self.buckets[self.hash(key)]
        while cur.next and cur.next.key != key:
            cur = cur.next
        if cur.next:
            cur.next = cur.next.next
            self.size -= 1

    def contains(self, key: int) -> bool:
        cur = self.buckets[self.hash(key)]
        while cur.next:
            if cur.next.key == key:
                return True
            cur = cur.next
        return False
        

# Your MyHashSet object will be instantiated and called as such:
# obj = MyHashSet()
# obj.add(key)
# obj.remove(key)
# param_3 = obj.contains(key)
